# 03d · RAG report: retrieval + generation results, figures, paper numbers

* Retrieval: from `results/rag/retrieval.json` (notebook 03b).
* Generation: from `03_rag/answers/answers.jsonl` (one answer per prompt in `generation_inputs.jsonl`) and
  `03_rag/answers/judgments.jsonl` (scores for each answer). Scoring rubric, applied to every answer the same way:
  - **faithfulness** = supported claims / factual claims, where a claim is supported if the retrieved passages given
    to that system state it (RAG systems only; L0 has no passages);
  - **correctness** = 1 if the answer contains the gold answer's key facts without contradicting them, 0.5 if
    partly, 0 if not;
  - **relevance** = 1 if the answer addresses the grower's question directly, 0.5 if partly, 0 if not.
* Independent rule-based check (no judge): PlantIQ's dose audit on every answer = share of answers stating a dose
  that is not in the passages the system was given.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "03_rag"))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common import config, env
from common import rag_eval as R
from plantiq_rag_snapshot.query import audit_dosages

config.ensure_dirs()
ANS_DIR = config.ROOT / ("_smoke/03_rag/answers" if config.SMOKE else "03_rag/answers")
ret = json.loads((config.RAG_RESULTS / "retrieval.json").read_text())
ABL = ["Dense only", "BM25 only", "Dense+BM25 (RRF)", "RRF + rerank", "Full PlantIQ (+topic, MMR)"]
SKR = "Selva Kumar et al. (dense, LangChain)"
SYS = {"PQ": "PlantIQ hybrid RAG", "SK": "Selva Kumar et al. (dense LangChain RAG)", "L0": "LLM, no retrieval"}

## Retrieval

In [ ]:
rs = pd.DataFrame(ret["summary"]).T
print(f"{ret['n_questions']} questions")
rs[["recall@1", "recall@3", "recall@5", "recall@10", "hit@3", "ndcg@3", "mrr@10"]]

In [ ]:
pd.DataFrame(ret.get("by_lang", {})).T

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.4))
labels = [SKR] + ABL
short = ["Selva Kumar\n(dense)", "Dense", "BM25", "Dense+BM25\n(RRF)", "RRF+\nrerank", "Full\n(+topic, MMR)"]
x = np.arange(len(labels))
for off, m, c in ((-0.18, "recall@3", "#4c78a8"), (0.18, "ndcg@3", "#f58518")):
    ax.bar(x + off, [ret["summary"][s][m] for s in labels], 0.36, label=m, color=c)
ax.set_xticks(x, short, fontsize=8)
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.set_title(f"Retrieval on {ret['n_questions']} questions (gold-evidence relevance)", fontsize=9)
plt.tight_layout()
for ext in ("png", "pdf"):
    plt.savefig(config.FIG_DIR / f"fig_rag_ablation.{ext}", dpi=200)
plt.show()

## Generation (needs answers.jsonl and judgments.jsonl)

In [ ]:
def read_jsonl(p):
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []

inputs = {r["key"]: r for r in read_jsonl(ANS_DIR / "generation_inputs.jsonl")}
answers = {r["key"]: r for r in read_jsonl(ANS_DIR / "answers.jsonl")}
judg = {r["key"]: r for r in read_jsonl(ANS_DIR / "judgments.jsonl")}
missing = [k for k in inputs if k not in answers or k not in judg]
gen = None
if not inputs:
    print("generation_inputs.jsonl not found: run 03c first.")
elif missing:
    print(f"{len(missing)} of {len(inputs)} prompts have no answer/judgment yet (first: {missing[:3]}). "
          "Generation results are skipped until both files are complete.")
else:
    rows = []
    for k, inp in inputs.items():
        a, j = answers[k], judg[k]
        audit = audit_dosages(a["answer"], [{"text": c} for c in inp["contexts"]])
        claims = int(j.get("claims", 0) or 0)
        rows.append({"key": k, "id": inp["id"], "system": inp["system"],
                     "faithfulness": (j["supported"] / claims) if (claims and inp["contexts"]) else np.nan,
                     "correctness": float(j["correct"]), "relevance": float(j["relevant"]),
                     "has_unsupported_dose": int(bool(audit["unverified_dosages"])),
                     "doses_in_answer": len(audit["detected_dosages"]), "words": len(a["answer"].split())})
    gen = pd.DataFrame(rows)
    gen.to_csv(config.RAG_RESULTS / "generation_rows.csv", index=False)
    M_ = ["faithfulness", "correctness", "relevance"]
    summary = {s: {"name": SYS[s], **{m: R.bootstrap_mean_ci(gen.loc[gen.system == s, m]) for m in M_},
                   "unsupported_dose_rate": float(gen.loc[gen.system == s, "has_unsupported_dose"].mean()),
                   "answers_with_doses": float((gen.loc[gen.system == s, "doses_in_answer"] > 0).mean()),
                   "mean_words": float(gen.loc[gen.system == s, "words"].mean())} for s in SYS}
    pq = gen[gen.system == "PQ"].set_index("id")
    tests = {s: {m: R.wilcoxon_paired(pq[m], gen[gen.system == s].set_index("id").reindex(pq.index)[m]) for m in M_}
             for s in ("SK", "L0")}
    meta_a = next(iter(answers.values()))
    env.write_json(config.RAG_RESULTS / "generation.json", {
        "systems": summary, "wilcoxon_pq_vs": tests, "n_prompts": len(gen),
        "generator": meta_a.get("generator"), "judge": next(iter(judg.values())).get("judge"),
        "rubric": "faithfulness = supported/claims vs retrieved passages; correctness and relevance 0/0.5/1 vs gold",
        "meta": env.run_meta()})
    table = pd.DataFrame({v["name"]: {**{m: v[m]["mean"] for m in M_}, "unsupported doses": v["unsupported_dose_rate"],
                                      "answers with doses": v["answers_with_doses"], "words": v["mean_words"]}
                          for v in summary.values()}).T
    display(table)
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    order, colors, w = ["PQ", "SK", "L0"], {"PQ": "#e6550d", "SK": "#6baed6", "L0": "#bdbdbd"}, 0.26
    for i, s in enumerate(order):
        vals = [np.nan_to_num(summary[s][m]["mean"]) for m in M_]
        err = np.nan_to_num(np.array([[summary[s][m]["mean"] - summary[s][m]["lo"],
                                       summary[s][m]["hi"] - summary[s][m]["mean"]] for m in M_]).T)
        ax.bar(np.arange(len(M_)) + (i - 1) * w, vals, w, yerr=err, capsize=2, label=SYS[s], color=colors[s])
    ax.set_xticks(range(len(M_)), ["Faithfulness", "Correctness", "Relevance"])
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
    ax.set_title("Generation quality (same LLM for every system; 95% bootstrap CI)", fontsize=9)
    plt.tight_layout()
    for ext in ("png", "pdf"):
        plt.savefig(config.FIG_DIR / f"fig_generation.{ext}", dpi=200)
    plt.show()

## Save paper numbers (same layout as IEEE-Paper/scripts/mock_data.json)

In [ ]:
paper = {"rag_ablation": {"is_mock": False, "configs": ["Dense only", "BM25 only", "Dense+BM25 (RRF)", "RRF + rerank",
                                                        "Full (+topic, MMR)"],
                          "Recall@3": [round(ret["summary"][c]["recall@3"], 4) for c in ABL],
                          "nDCG@3": [round(ret["summary"][c]["ndcg@3"], 4) for c in ABL],
                          "selva_kumar_dense": {"Recall@3": round(ret["summary"][SKR]["recall@3"], 4),
                                                "nDCG@3": round(ret["summary"][SKR]["ndcg@3"], 4)},
                          "plantiq_router_offline": {k: round(ret["summary"].get("Full PlantIQ, router offline", {}).get(k, float("nan")), 4)
                                                     for k in ("recall@3", "ndcg@3")},
                          "by_lang": ret.get("by_lang", {})}}
if gen is not None:
    paper["generation_comparison"] = {"is_mock": False, "metrics": ["Faithfulness", "Correctness", "Relevance"],
                                      "systems": {summary[s]["name"]: [round(float(np.nan_to_num(summary[s][m]["mean"])), 4)
                                                                       for m in ("faithfulness", "correctness", "relevance")]
                                                  for s in ("PQ", "SK", "L0")}}
    paper["table_viii"] = [{"system": summary[s]["name"],
                            "recall@3": round(ret["summary"]["Full PlantIQ (+topic, MMR)" if s == "PQ" else SKR]["recall@3"], 3)
                            if s != "L0" else None,
                            "faithfulness": None if s == "L0" else round(summary[s]["faithfulness"]["mean"], 3),
                            "correctness": round(summary[s]["correctness"]["mean"], 3),
                            "unsupported_doses_pct": round(100 * summary[s]["unsupported_dose_rate"], 1)}
                           for s in ("L0", "SK", "PQ")]
env.write_json(config.RAG_RESULTS / "paper_rag.json", paper)
md = ["## Module 2 (RAG)", "", f"{ret['n_questions']} questions (English + Kannada) for retrieval.", "",
      "### Retrieval", "", rs.to_markdown(floatfmt=".3f"), ""]
if gen is not None:
    md += ["### Generation (same LLM for every system)", "", table.to_markdown(floatfmt=".3f"), ""]
(config.RESULTS_DIR / "RESULTS_rag.md").write_text("\n".join(md), encoding="utf-8")
print("\n".join(md))